# 27b · Input ablation multi-view — MIMIC-CXR `cohort_v2` (13 nhãn)

> **Người nhận bundle:** mục **0–1** là đủ để bắt đầu · **2–4** là hợp đồng dữ liệu và cách
> gắn vào model · **5–7** là những chỗ dễ vấp, đọc trước khi tin kết quả.
> Mục **8** chỉ dành cho người chạy notebook này.

Notebook này **không train gì cả**, **không gắn vào RAD-DINO**, và **không cần GPU**. Nó
chỉ dựng và đóng gói *đầu vào* cho 6 arm multi-view, để cắm vào kiến trúc bất kỳ.

---

## 0 · Ai chạy cái gì

| vai | làm gì |
|---|---|
| **Người chạy notebook này** (có dữ liệu MIMIC-CXR + `planB_out/`) | chạy hết `C0 → C7`, được `mv_input/` + `mv_input.zip`, gửi file zip đi |
| **Người nhận bundle** (có kiến trúc riêng) | giải nén, đọc `README.md`, dùng `mv_pool.py`. **Không cần chạy notebook này.** |

Nếu bạn là người nhận: mọi thứ bạn cần nằm trong zip. Notebook này chỉ giải thích bundle
đó *được dựng ra thế nào* — đọc mục 2–7 để hiểu hợp đồng, rồi bỏ qua phần code.

---

## 1 · Bắt đầu trong 30 giây 

Bạn chạy backbone **của bạn** trên từng ảnh, theo đúng thứ tự `image_manifest.parquet`:

```python
man = pd.read_parquet('image_manifest.parquet')
F   = np.stack([backbone(load(d)) for d in man.dicom_id])   # [n_img, d]
```

Rồi mỗi arm là **một dòng**:

```python
z    = np.load('arm_weights.npz')
x    = np.add.reduceat(F * z['w_P73'][:, None], z['offsets'][:-1], axis=0)   # [n_study, d]
```

Đổi `w_P73` → `w_P64` / `w_P55` / `w_FRONTAL` / `w_MEAN` là ra arm khác. Trọng số đã tổng
bằng 1 trên từng study nên **không cần chuẩn hoá lại**.

**Thứ tự hàng là thứ duy nhất quan trọng.** Sai thứ tự thì mọi thứ sau đều sai *mà không
báo lỗi*. `image_manifest.parquet` là thứ tự chủ của ảnh; `study_index.parquet` là thứ tự
chủ của study; `Y` trong `labels.npz` theo thứ tự study.

---

## 2 · Sáu arm

| arm | pooling | xuất ra dưới dạng |
|---|---|---|
| `P73` | `w_f=0.7 / w_l=0.3` — default hiện hành của `05` | vector trọng số per-image |
| `P64` | `0.6 / 0.4` | vector trọng số per-image |
| `P55` | `0.5 / 0.5` | vector trọng số per-image |
| `FRONTAL` | chỉ frontal | vector trọng số per-image |
| `MEAN` | trung bình đều mọi ảnh (**control** của `ATTN`) | vector trọng số per-image |
| `ATTN` | **gated attention** — học được | bó ragged + module mẫu |

`MEAN` không nằm trong 5 arm được yêu cầu ban đầu. Nó có mặt vì là **null đúng** của
`ATTN`: module attention init `w=0` khởi động *bằng* `MEAN`, nên `ATTN − MEAN` là phần
attention thực sự học được, đã tách khỏi phần chỉ do thêm tham số. Đừng bỏ nó đi.

---

## 3 · Vì sao là *trọng số*, không phải embedding

Bốn arm cố định đều có chung một dạng: feature study-level là **tổ hợp tuyến tính của
feature từng ảnh trong study**

$$x_{\text{study}}=\sum_{i\in\text{study}} w_i\,f_i,\qquad \sum_{i\in\text{study}} w_i = 1$$

`w` **không phụ thuộc backbone** — nó chỉ phụ thuộc study đó có mấy ảnh frontal, mấy ảnh
lateral. Nên thứ đáng giao đi là `w`, chứ không phải một đống embedding RAD-DINO mà bên
nhận sẽ vứt đi.

`C5` xuất thêm bản pool sẵn từ cache CLS của `05` (cờ `EXPORT_EMBEDDINGS`, mặc định
**tắt**), phòng khi bên nhận muốn dùng luôn feature RAD-DINO. Bật thì bundle từ ~50 MB lên
~1.7 GB và dính chặt vào RAD-DINO — chỉ bật nếu có lý do rõ ràng.

### Bundle có gì

| file | nội dung |
|---|---|
| `study_index.parquet` | 1 dòng / study, **thứ tự chủ**. `study_row` là chỉ số dùng ở mọi mảng khác. |
| `image_manifest.parquet` | 1 dòng / ảnh, **liên tục theo study**. `row` = chỉ số hàng trong chính file này. |
| `arm_weights.npz` | `w_P73`, `w_P64`, `w_P55`, `w_FRONTAL`, `w_MEAN` mỗi cái `[n_img]`; `offsets [n_study+1]`. |
| `ragged.npz` | `pad_idx [N,K] int32`, `pad_mask`, `pad_is_frontal` — cho arm `ATTN`. |
| `labels.npz` | `Y [N,13] float32`, `labels`, `tier`, `train_prev`. |
| `mv_pool.py` | module drop-in: `MultiViewPool`, `collate_studies`, `StudyModel`. |
| `manifest.json` · `README.md` | số liệu tóm tắt + hợp đồng đầy đủ. |
| `emb/` | *(tuỳ chọn)* feature RAD-DINO đã pool sẵn. |

---

## 4 · Gắn vào model: chỗ khó là **batching**, không phải pooling

Dataloader thường trả một *ảnh* một sample; multiview cần một *study* một sample, với số
ảnh thay đổi từng study. `mv_pool.py` chọn **phẳng + segment id** thay vì đệm:

```
batch 64 study  ->  M ~ 105 anh (trung binh 1.64 anh/study)
  dem [B,K,3,H,W] toi K=8 :  512 anh  ->  phi ~79% compute cua backbone
  phang + segment id      :  105 anh  ->  khong phi gi
```

Backbone chạy **một lần** trên `M` ảnh phẳng, rồi `index_add_` gom về `[B, d]`:

```python
from mv_pool import MultiViewPool, collate_studies, StudyModel

model = StudyModel(your_backbone, d=768, n_labels=13, mode='fixed')
imgs, seg, vf, w, y, B = batch            # tu collate_studies
logits = model(imgs, seg, B, vf=vf, w=w)
```

**Đổi arm = đổi vector `w` truyền vào. Model không đổi một dòng nào.** Đó chính là lý do
bundle xuất trọng số *per-image*: nó khớp thẳng với batching phẳng, không phải reindex gì.
Arm `ATTN`: `mode='attn'`, bỏ `w`, truyền `vf`.

Nếu backbone **đóng băng**: thay `images` bằng `m.row.values` và dùng
`FrozenFeatureBackbone(F)` — lúc đó cả ablation chạy CPU cũng được, mỗi arm vài phút.

### Gắn ở tầng nào

| tầng | đường đi | ghi chú |
|---|---|---|
| **embedding** *(bundle giả định)* | `backbone → f → pool → head` | head nhìn vector đã pool. Là câu hỏi **chưa ai đo**. |
| **xác suất** | `backbone → head → p mỗi ảnh → pool p` | áp **đúng `w` đó** lên `p`. Không đổi model. Đã đo, kết quả null — mục 7. |
| **token** | ghép patch token đa view thành một chuỗi → attention | mạnh nhất, đắt nhất. |

Cùng bộ `w` dùng được ở **cả** mức embedding lẫn mức xác suất, vì cả hai đều là tổ hợp lồi
tổng bằng 1. Muốn đo rẻ trước thì đi đường xác suất — không phải sửa model gì cả.

### Một ràng buộc, không phải gợi ý

Score attention **phải đọc cờ frontal/lateral**. Có cờ thì `ATTN` biểu diễn được cả 5 arm
cố định (chỉ cần học một hằng số theo view) ⇒ siêu tập đúng nghĩa, so sánh mới có ý nghĩa.
Không cờ thì nó là một họ khác hẳn và Δ vô nghĩa. `mv_pool.py` đã làm sẵn.

---

## 5 · Tám chỗ dễ sai

1. **Chuẩn hoá feature ở mức TỪNG ẢNH, không ở mức study.** Với trọng số tổng bằng 1,
   `pool(standardize(f)) = standardize(pool(f))` **chính xác**, nên mọi arm chung một thang
   và không arm nào thiệt vì scale. Chuẩn hoá *sau* khi pool thì mỗi arm phải fit
   `mean/std` riêng, và bạn mất khả năng phân biệt "arm tốt hơn" với "arm hợp LR hơn".
   `mean/std` lấy từ ảnh của study **train**.
2. **`FRONTAL` và 366 study chỉ có lateral.** Xem `frontal_is_fallback` trong
   `study_index.parquet`. Mặc định các study đó dùng lateral mean để giữ N không đổi giữa
   các arm. Muốn theo quy ước `04b` (loại hẳn) thì lọc cột đó ở **cả 5 arm** — không thì
   `FRONTAL` được chấm trên tập study khác và Δ không còn quy về pooling.
3. **Chia split theo `subject_id`, không theo study.** Cột `split` có sẵn đã làm đúng; nếu
   chia lại thì phải giữ ràng buộc, không thì ảnh cùng bệnh nhân rơi vào cả train lẫn test.
4. **Đừng so arm bằng một seed.** Chênh lệch giữa các cách pooling ở bộ này rất nhỏ. Chạy
   ≥ 3 seed và so Δ với độ lệch chuẩn giữa các seed trước — Δ nhỏ hơn nhiễu train thì
   không phải phát hiện.
5. **`batch_size` giờ đếm STUDY, không đếm ảnh.** Backbone thực nhận `M ≈ 1.64 × B` ảnh
   (tối đa 8×). Nếu trước đó bạn batch theo ảnh thì phải hạ `B` xuống ~60%, không thì OOM
   ngay batch đầu.
6. **Đừng để ảnh của cùng một study rơi vào hai batch.** `collate_studies` lo việc đó nếu
   Dataset trả theo study; tự viết sampler thì phải giữ ràng buộc này.
7. **BatchNorm trong backbone.** Batch bây giờ là *ảnh*, nên study nhiều ảnh "nặng" hơn
   trong thống kê BN — một confound nhỏ giữa các arm nếu backbone train cùng.
   LayerNorm/GroupNorm không dính.
8. **TTA: không lật ảnh.** Lật ngang đổi bên tim/phổi — với CXR đó là thay đổi bệnh lý,
   không phải augmentation. Chỉ dùng biến thiên thu nhận thực tế (xoay nhỏ, crop/scale).

---

## 6 · `ATTN` không precompute được — và đó không phải thiếu sót

Bốn arm kia là hằng số. Attention thì trọng số **do model sinh ra từ chính `f`**, nên nó
chỉ tồn tại lúc forward. Thứ duy nhất precompute được là *cấu trúc ragged*: ảnh nào thuộc
study nào, ảnh nào là frontal. Đó là cái `ragged.npz` chứa.

---

## 7 · Bối cảnh phải biết trước khi đọc kết quả

Đo ở **mức xác suất** (model đóng băng, chỉ đổi cách trộn đầu ra), pooling học được từng
cho kết quả **null**:

```
V3_learned_pooler - moc   dMacro +0.00047  CI95 [-0.00096, +0.00199]  chua 0
-> POOLING KHONG phai nut that.
```

Ablation mà bundle này phục vụ ở **mức embedding** (pool trước head, head train lại) là
câu hỏi *khác* và chưa ai đo. Nhưng con số trên là **kỳ vọng cơ sở**: dư địa ở trục này
rất mỏng. Nếu kết quả của bạn cũng có CI chứa 0 thì hai mức đã đồng thuận, và trục
multi-view nên đóng lại — chứ không phải chạy tiếp biến thể thứ ba.

---

## 8 · Chạy notebook này

**Cần:** `03` xong (`view_map.parquet`), `02` xong (`cohort_study_level.parquet`,
`class_freq.pkl`, `cohort_meta.json`). Cache CLS của `05` **chỉ cần khi** bật
`EXPORT_EMBEDDINGS`.

**Cấu hình** — tất cả ở `C0`: `FIXED_ARMS`, `INCLUDE_MEAN_CONTROL`,
`FRONTAL_LATERAL_FALLBACK`, `K_MAX`, `EXPORT_EMBEDDINGS`, `MAKE_ZIP`.

**Phần cứng:** CPU thuần, **không cần GPU**. Toàn bộ là numpy; `C6-bis` có dùng torch
nhưng chỉ để tự kiểm module.

| chế độ | RAM đỉnh | thời gian |
|---|---|---|
| `EXPORT_EMBEDDINGS=False` (mặc định) | < 1 GB | ~30–60 s |
| `EXPORT_EMBEDDINGS=True` | ~3 GB | ~4–6 phút (đọc 1.06 GB cache từ Drive là phần chậm) |

### Bản đồ cell

| cell | làm gì | cổng |
|---|---|---|
| `C0` | config, kiểm prereq | — |
| `C1` | cohort, nhãn, split, tier | — |
| `C2` | manifest ảnh, liên tục theo study | composition phải phủ hết |
| `C3` | trọng số per-image từng arm | `sum(w) = 1` trên **mọi** study |
| `C4` | bó ragged cho `ATTN` | `pad_idx` không trỏ sang study khác; lateral sống > 99% |
| `C5` | *(tuỳ chọn)* pool cache CLS của `05` | `P73` tái lập `study_x_cxr.npy` ≤ 1e-3 |
| `C6` | ghi bundle + `README.md` | — |
| `C6-bis` | ghi `mv_pool.py` | exec module, chạy thật 3 phép kiểm |
| `C7` | nạp lại từ đĩa, dựng lại 5 arm | round-trip; ragged pool đều == `MEAN` |

**Cổng đáng chạy ít nhất một lần:** `C5` với `EXPORT_EMBEDDINGS=True`. Nó tái lập
`study_x_cxr.npy` mà `05` đã ghi — bắt được **cả** sai công thức trọng số **lẫn** sai thứ
tự manifest, cùng lúc. Qua cổng đó rồi hãy tắt đi và xuất bản gửi.

---

*Notebook này thuộc nhánh `cohort_v2` (01–27), không phải nhánh CXR-LT (28+).*

## C0 · Config

In [ ]:
import os, json, pickle, shutil
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('PLANB_OUT', './planB_out')

OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
CACHE_DIR = os.path.join(DATA_DIR, 'cxr_cls_cache')      # chi dung khi EXPORT_EMBEDDINGS
EXP_DIR   = os.path.join(DATA_DIR, 'mv_input')           # <- thu muc se gui di
os.makedirs(EXP_DIR, exist_ok=True)

# ten arm -> w_f. w_l = 1 - w_f. ATTN khong o day vi khong precompute duoc.
FIXED_ARMS = {'P73': 0.7, 'P64': 0.6, 'P55': 0.5, 'FRONTAL': 1.0}
INCLUDE_MEAN_CONTROL     = True     # them arm MEAN = trung binh deu tren moi anh
FRONTAL_LATERAL_FALLBACK = True     # xem header. False -> danh dau keep=False cho study lateral-only
K_MAX          = 8                  # tran so anh/study trong bo ragged cua ATTN
EXPORT_EMBEDDINGS = False           # True -> pool luon cache CLS cua 05 thanh x_<arm>.npy [N,768]
MAKE_ZIP          = True            # dong goi EXP_DIR thanh mot file .zip de gui

for _f in ['view_map.parquet', 'cohort_study_level.parquet', 'class_freq.pkl', 'cohort_meta.json']:
    assert os.path.exists(os.path.join(OUT_DIR, _f)), f'{_f} khong co trong {OUT_DIR}'
if EXPORT_EMBEDDINGS:
    for _f in ['cxr_cls.npy', 'cxr_cls_index.parquet']:
        assert os.path.exists(os.path.join(CACHE_DIR, _f)), \
            f'{_f} khong co -> chay 05 truoc, hoac dat EXPORT_EMBEDDINGS=False'

ARM_NAMES = list(FIXED_ARMS) + (['MEAN'] if INCLUDE_MEAN_CONTROL else []) + ['ATTN']
print('arms      :', ARM_NAMES)
print('export -> ', EXP_DIR)
print('embeddings:', 'CO (can cache 05)' if EXPORT_EMBEDDINGS else 'KHONG (chi trong so, agnostic backbone)')

## C1 · Cohort · nhãn · split

`ehr` sort theo `study_id` là **trật tự chủ**: mọi mảng study-level xuất ra (`Y`, trọng số,
ragged) đều theo trật tự này, và `study_index.parquet` ghi lại nó để bên nhận không phải
đoán.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13
C = len(LABELS)

ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'))
ehr = ehr.sort_values('study_id').reset_index(drop=True)      # TRAT TU CHU
N   = len(ehr)
SID = ehr.study_id.to_numpy()
Y   = ehr[LABELS].to_numpy('float32')
split = ehr.split.to_numpy()

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
assert cf.get('labels', LABELS) == LABELS, 'class_freq label order != cohort'
TRAIN_PREV = np.asarray(cf['class_freq']) / cf['num_train']
TIER = np.where(TRAIN_PREV < 0.02, 'tail', np.where(TRAIN_PREV < 0.10, 'medium', 'head'))

print(f'cohort: {N:,} studies x {C} labels')
print('splits:', {s: int((split == s).sum()) for s in ['train', 'val', 'test']})
print(pd.DataFrame({'label': LABELS, 'tier': TIER, 'train_prev': TRAIN_PREV.round(4)})
        .sort_values('train_prev').to_string(index=False))

## C2 · Manifest ảnh — liên tục theo study

Xếp mọi ảnh frontal/lateral thành **một bảng liên tục theo study** (ảnh của cùng study nằm
cạnh nhau, trong study sắp theo `dicom_id`). Bố cục này là thứ khiến pooling chỉ còn một
lệnh `np.add.reduceat` — và cũng là thứ bên nhận cần biết để gom feature của họ đúng thứ tự.

`row` trong manifest là **chỉ số hàng trong chính manifest**, không phải chỉ số trong cache
của `05`. Bên nhận chạy backbone theo đúng thứ tự `dicom_id` của manifest là khớp.

In [ ]:
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])].copy()

sid_pos = pd.Series(np.arange(N), index=SID)
vm = vm[vm.study_id.isin(sid_pos.index)].copy()
vm['study_row'] = vm.study_id.map(sid_pos).to_numpy()
IMG = (vm.sort_values(['study_row', 'dicom_id'])
         .reset_index(drop=True)[['study_row', 'study_id', 'dicom_id', 'view_group']])
IMG.insert(0, 'row', np.arange(len(IMG)))
IMG['pos_in_study'] = IMG.groupby('study_row').cumcount()

srow     = IMG.study_row.to_numpy()
is_front = (IMG.view_group.to_numpy() == 'frontal')
n_img    = np.bincount(srow, minlength=N)
n_f      = np.bincount(srow[is_front],  minlength=N)
n_l      = np.bincount(srow[~is_front], minlength=N)
assert n_img.min() >= 1, 'co study khong con anh frontal/lateral nao -> kiem tra lai 01/03'
assert int(n_img.sum()) == len(IMG)
OFFS = np.concatenate([[0], np.cumsum(n_img)])          # OFFS[i]:OFFS[i+1] = anh cua study i

comp = np.select([(n_f > 0) & (n_l > 0), (n_f > 0) & (n_l == 0), (n_f == 0) & (n_l > 0)],
                 ['both', 'frontal_only', 'lateral_only'], 'none')
print(f'images {len(IMG):,} | studies {N:,} | anh/study: tb {n_img.mean():.2f}, max {n_img.max()}')
print('view      :', {'frontal': int(is_front.sum()), 'lateral': int((~is_front).sum())})
print('composition:', dict(pd.Series(comp).value_counts()))
assert (comp != 'none').all()

## C3 · Trọng số per-image cho từng arm cố định

Với arm `w_f`, ảnh thứ `i` của study `s` nhận

$$w_i=\frac{w_{\text{view}(i)}\,/\,n_{\text{view}(i)}(s)}{w_f\,[\exists F]+w_l\,[\exists L]}$$

Chia cho `n_view` là bước "mean **trong** view trước, weight **giữa** view sau" — để study
có 3 ảnh frontal không lấn át study có 1. Mẫu số là tái chuẩn hoá của `05`/`04b`: 51.6%
study chỉ có frontal, nếu để tổng có trọng số trần thì embedding của chúng bị co lại 0.7
lần và trôi khỏi manifold của nhóm hai view.

### Cổng — trọng số phải tổng bằng 1 trên **mọi** study

Nếu cổng này trượt thì có arm đang lặng lẽ đổi scale của một nhóm study, và mọi so sánh
sau đó là so scale chứ không phải so pooling.

In [ ]:
den_f = np.where(n_f > 0, n_f, 1).astype('float32')
den_l = np.where(n_l > 0, n_l, 1).astype('float32')
has_f = (n_f > 0).astype('float32')
has_l = (n_l > 0).astype('float32')

lat_only = (n_f == 0)
print(f'study chi co lateral: {int(lat_only.sum()):,} ({lat_only.mean():.3%})')

def arm_weights(w_f):
    """[n_img] float32, tong = 1 tren tung study."""
    w_l = 1.0 - w_f
    norm = w_f * has_f + w_l * has_l                     # mau so tai chuan hoa
    fb = np.zeros(N, bool)
    if w_f >= 1.0 - 1e-9:                                # FRONTAL: mau so = 0 o study lateral-only
        fb = lat_only.copy()
        if FRONTAL_LATERAL_FALLBACK:
            norm = np.where(fb, w_l if w_l > 0 else 1.0, norm)
        else:
            norm = np.where(fb, 1.0, norm)               # tranh chia 0; study nay se bi keep=False
    per_img = np.where(is_front[np.arange(len(IMG))],
                       w_f / den_f[srow], w_l / den_l[srow]).astype('float32')
    if w_f >= 1.0 - 1e-9 and FRONTAL_LATERAL_FALLBACK:   # 366 study do: dung lateral mean
        m = fb[srow] & ~is_front
        per_img = np.where(m, (1.0 / den_l[srow]).astype('float32'), per_img)
    w = per_img / norm[srow].astype('float32')
    if w_f >= 1.0 - 1e-9 and not FRONTAL_LATERAL_FALLBACK:
        w = np.where(fb[srow], 0.0, w).astype('float32')
    return w.astype('float32'), fb

W, FALLBACK = {}, np.zeros(N, bool)
for arm, w_f in FIXED_ARMS.items():
    W[arm], fb = arm_weights(w_f)
    if fb.any():
        FALLBACK = FALLBACK | fb
if INCLUDE_MEAN_CONTROL:
    W['MEAN'] = (1.0 / n_img[srow]).astype('float32')

KEEP = np.ones(N, bool) if FRONTAL_LATERAL_FALLBACK else ~lat_only

# ---------------- COÅNG: tong trong so = 1 tren tung study ----------------
print('\nCOÅNG · sum(w) tren tung study:')
for arm, w in W.items():
    s = np.add.reduceat(w, OFFS[:-1])
    bad = np.abs(s[KEEP] - 1.0).max()
    print(f'  {arm:8s} max|sum-1| = {bad:.2e}  {"OK" if bad <= 1e-5 else "FAIL"}')
    assert bad <= 1e-5, f'{arm}: trong so khong tong 1 -> arm nay dang doi scale study'
print(f'\nstudy dung fallback lateral cho FRONTAL: {int(FALLBACK.sum()):,}')
print(f'study dung de danh gia (keep): {int(KEEP.sum()):,} / {N:,}')

## C4 · Bó ragged cho `ATTN`

Attention cần truy cập từng ảnh của study, nên ta đệm chỉ mục thành `[N, K]` cộng một mask.
`pad_idx` trỏ vào **hàng của manifest** — nghĩa là bên nhận chỉ cần một mảng feature `F`
xếp đúng thứ tự manifest, rồi `F[pad_idx]` là ra tensor `[N, K, d]`.

Trần `K_MAX` chặn một study bất thường kéo cả batch phình ra. Cell in ra số study bị cắt và
**tỉ lệ lateral còn sống sót sau khi cắt** — vì cắt theo `dicom_id` mà lệch view thì `ATTN`
sẽ thua một cách giả tạo, và đó là loại lỗi không tự lộ ra.

In [ ]:
K = int(min(n_img.max(), K_MAX))
_ar   = np.arange(K)[None, :]
_take = _ar < np.minimum(n_img, K)[:, None]
_pos  = np.clip(OFFS[:-1][:, None] + _ar, 0, len(IMG) - 1)

PAD_IDX   = np.where(_take, _pos, 0).astype('int32')
PAD_MASK  = _take
PAD_FRONT = np.where(_take, is_front[_pos], False)

n_trunc   = int((n_img > K).sum())
lat_all   = int((~is_front).sum())
lat_kept  = int(np.where(_take, ~is_front[_pos], False).sum())
print(f'K = {K} | study bi cat: {n_trunc:,} ({n_trunc/N:.3%})')
print(f'lateral con lai sau khi cat: {lat_kept:,}/{lat_all:,} ({lat_kept/lat_all:.3%})')
assert lat_kept / lat_all > 0.99, 'cat theo dicom_id dang an mat lateral -> tang K_MAX'

# ---------------- COÅNG: mask khop so anh, pad_idx nam dung study ----------------
assert (PAD_MASK.sum(1) == np.minimum(n_img, K)).all()
_owner = np.where(PAD_MASK, srow[PAD_IDX], np.arange(N)[:, None])
assert (_owner == np.arange(N)[:, None]).all(), 'pad_idx tro sang study khac'
print('COÅNG · ragged nhat quan voi manifest: OK')

## C5 · (tuỳ chọn) Pool sẵn cache CLS của `05`

Chỉ chạy khi `EXPORT_EMBEDDINGS=True`. Đây là phần **duy nhất** dính tới RAD-DINO — bỏ qua
được mà bundle vẫn dùng đủ.

Kèm một cổng đáng giá: pool bằng trọng số `P73` phải tái lập `study_x_cxr.npy` mà `05` đã
ghi. Nếu khớp thì công thức trọng số ở `C3` **và** thứ tự manifest ở `C2` đều đúng — một
cổng bắt được cả hai loại lỗi cùng lúc.

*(CPU. ~4–6 phút, RAM đỉnh ~3 GB, phần lớn là đọc 1.06 GB từ Drive.)*

In [ ]:
def pool_with(Fmat, w):
    """[n_img, d] + [n_img] -> [N, d]. Manifest lien tuc theo study nen reduceat la du."""
    return np.add.reduceat(Fmat * w[:, None], OFFS[:-1], axis=0).astype('float32')

if EXPORT_EMBEDDINGS:
    emb_dir = os.path.join(EXP_DIR, 'emb'); os.makedirs(emb_dir, exist_ok=True)
    cidx = pd.read_parquet(os.path.join(CACHE_DIR, 'cxr_cls_index.parquet'))
    src  = pd.Series(cidx.row.to_numpy(), index=cidx.dicom_id.to_numpy())
    take = src.reindex(IMG.dicom_id.to_numpy())
    assert take.notna().all(), f'{int(take.isna().sum())} dicom trong manifest khong co trong cache 05'
    Fm = np.load(os.path.join(CACHE_DIR, 'cxr_cls.npy'))[take.to_numpy().astype('int64')]
    print(f'CLS per-image xep theo manifest: {Fm.shape}')

    for arm, w in W.items():
        Xa = pool_with(Fm, w)
        np.save(os.path.join(emb_dir, f'x_{arm}.npy'), Xa)
        print(f'  x_{arm}.npy {Xa.shape}  {Xa.nbytes/1e6:.0f} MB')

    _ref_p = os.path.join(CACHE_DIR, 'study_x_cxr.npy')
    if os.path.exists(_ref_p):
        _ri  = pd.read_parquet(os.path.join(CACHE_DIR, 'study_x_cxr_index.parquet'))
        _row = pd.Series(_ri.row.to_numpy(), index=_ri.study_id.to_numpy()).reindex(SID).to_numpy()
        _e   = float(np.abs(pool_with(Fm, W['P73']) - np.load(_ref_p)[_row]).max())
        print(f'COÅNG · P73 tai lap 05 study_x_cxr: max_abs_diff {_e:.2e}  '
              f'{"OK" if _e <= 1e-3 else "FAIL"}')
        assert _e <= 1e-3, 'trong so C3 hoac thu tu manifest C2 SAI -> dung lai'
    else:
        print('! khong co study_x_cxr.npy -> bo qua cong tai lap 05')
    np.save(os.path.join(emb_dir, 'cls_per_image.npy'), Fm)
    print(f'  cls_per_image.npy {Fm.shape}  {Fm.nbytes/1e9:.2f} GB  (theo thu tu manifest)')
    del Fm
else:
    print('EXPORT_EMBEDDINGS=False -> bo qua. Bundle chi gom trong so + ragged (agnostic backbone).')

## C6 · Ghi bundle

Bảy file, tất cả đều theo **trật tự chủ** đã khoá ở `C1`/`C2`. `README.md` đi kèm ghi rõ
hợp đồng và có sẵn đoạn code dùng thử, để bên nhận không phải hỏi lại thứ tự hàng.

In [ ]:
IMG.to_parquet(os.path.join(EXP_DIR, 'image_manifest.parquet'), index=False)

pd.DataFrame({
    'study_row': np.arange(N), 'study_id': SID, 'split': split,
    'n_img': n_img, 'n_frontal': n_f, 'n_lateral': n_l,
    'composition': comp, 'keep': KEEP, 'frontal_is_fallback': FALLBACK,
}).to_parquet(os.path.join(EXP_DIR, 'study_index.parquet'), index=False)

np.savez_compressed(os.path.join(EXP_DIR, 'arm_weights.npz'),
                    offsets=OFFS.astype('int64'), **{f'w_{a}': w for a, w in W.items()})
np.savez_compressed(os.path.join(EXP_DIR, 'ragged.npz'),
                    pad_idx=PAD_IDX, pad_mask=PAD_MASK, pad_is_frontal=PAD_FRONT, K=np.int32(K))
np.savez_compressed(os.path.join(EXP_DIR, 'labels.npz'),
                    Y=Y, labels=np.array(LABELS), tier=TIER, train_prev=TRAIN_PREV)
json.dump(dict(n_studies=int(N), n_images=int(len(IMG)), n_labels=C, K=int(K),
               arms=ARM_NAMES, fixed_arms=FIXED_ARMS,
               frontal_lateral_fallback=bool(FRONTAL_LATERAL_FALLBACK),
               n_lateral_only=int(lat_only.sum()), n_truncated=int(n_trunc),
               splits={s: int((split == s).sum()) for s in ['train', 'val', 'test']},
               source='MIMIC-CXR cohort_v2, view_map tu nb03, labels tu nb02',
               embeddings_included=bool(EXPORT_EMBEDDINGS)),
          open(os.path.join(EXP_DIR, 'manifest.json'), 'w'), indent=1)
print('da ghi 6 file du lieu')

### `README.md` — hợp đồng gửi kèm

Viết bằng `Write`-style một lần, không nối chuỗi lắt nhắt, để nội dung file khớp đúng cái
đọc được ở đây.

In [ ]:
README = """# Input ablation multi-view — MIMIC-CXR cohort_v2 (13 nhan)

Bo nay KHONG chua model va KHONG buoc vao backbone nao. No dinh nghia cach gop
nhieu anh (frontal/lateral) cua mot study thanh mot feature study-level, theo 6 arm.

## Files

| file | noi dung |
|---|---|
| `study_index.parquet` | 1 dong / study, theo TRAT TU CHU. `study_row` la chi so hang dung o moi mang khac. |
| `image_manifest.parquet` | 1 dong / anh, LIEN TUC theo study. `row` = chi so hang trong chinh file nay. |
| `arm_weights.npz` | `w_P73`, `w_P64`, `w_P55`, `w_FRONTAL`, `w_MEAN`: moi cai `[n_img]`. `offsets` `[n_study+1]`. |
| `ragged.npz` | `pad_idx [N,K] int32`, `pad_mask [N,K] bool`, `pad_is_frontal [N,K] bool` — cho arm ATTN. |
| `labels.npz` | `Y [N,13] float32`, `labels`, `tier`, `train_prev`. |
| `manifest.json` | so lieu tom tat + cau hinh da dung. |
| `mv_pool.py` | module drop-in: `MultiViewPool`, `collate_studies`, `StudyModel`. Xem "Buoc 3". |
| `emb/` | (tuy chon) `cls_per_image.npy` + `x_<arm>.npy` — feature RAD-DINO da pool san. |

## Buoc 1 — sinh feature per-image bang backbone CUA BAN

Chay backbone tren tung anh theo DUNG thu tu `image_manifest.parquet`:

    man = pd.read_parquet('image_manifest.parquet')
    F = np.stack([backbone(load(d)) for d in man.dicom_id])   # [n_img, d]

Thu tu la thu quan trong duy nhat. Sai thu tu thi moi thu sau deu sai ma khong bao loi.

## Buoc 2a — 5 arm trong so co dinh: mot dong

    z = np.load('arm_weights.npz')
    offs = z['offsets']
    x = np.add.reduceat(F * z['w_P73'][:, None], offs[:-1], axis=0)   # [n_study, d]

Doi `w_P73` thanh `w_P64` / `w_P55` / `w_FRONTAL` / `w_MEAN` de duoc arm khac.
Trong so da tong bang 1 tren tung study, nen khong can chuan hoa lai.

Ban PyTorch (giu duoc gradient neu backbone train cung):

    seg = torch.repeat_interleave(torch.arange(N), torch.tensor(n_img))
    x = torch.zeros(N, d).index_add_(0, seg, F * w[:, None])

## Buoc 2b — arm ATTN: pooling hoc duoc

    r = np.load('ragged.npz')
    E    = F[r['pad_idx']]                      # [N, K, d]
    mask = torch.as_tensor(r['pad_mask'])       # [N, K]
    vf   = torch.as_tensor(r['pad_is_frontal']).float()

    class GatedAttnPool(nn.Module):
        # Ilse et al. 2018 (ABMIL). Score doc them co view -> bieu dien duoc
        # CA 5 arm co dinh, nen so sanh moi co nghia (sieu tap dung nghia).
        def __init__(self, d, h=128):
            super().__init__()
            self.V = nn.Linear(d + 2, h); self.U = nn.Linear(d + 2, h)
            self.w = nn.Linear(h, 1)
            nn.init.zeros_(self.w.weight); nn.init.zeros_(self.w.bias)  # init == arm MEAN
        def forward(self, E, mask, vf):
            z = torch.cat([E, vf.unsqueeze(-1), 1 - vf.unsqueeze(-1)], -1)
            s = self.w(torch.tanh(self.V(z)) * torch.sigmoid(self.U(z))).squeeze(-1)
            a = torch.softmax(s.masked_fill(~mask, float('-inf')), 1)
            return (a.unsqueeze(-1) * E).sum(1), a

`w` init bang 0 nen attention KHOI DONG dung bang arm MEAN va phai tu kiem moi sai lech.
Giu nguyen diem nay: no la cai khien "ATTN - MEAN" doc duoc nhu phan attention hoc them,
tach khoi phan chi do them tham so.

## Buoc 3 — gan vao model cua ban: dung `mv_pool.py`

Cho khó không phai cong thuc pooling, ma la BATCHING: dataloader thuong tra mot ANH
mot sample, multiview can mot STUDY mot sample voi so anh thay doi.

`mv_pool.py` chon cach PHANG + segment id thay vi dem `[B, K, 3, H, W]`:

    batch 64 study -> M ~ 105 anh (trung binh 1.64 anh/study)
      dem toi K=8 : 512 anh  -> lang phi ~79% compute cua backbone
      phang       : 105 anh  -> khong lang phi gi

Backbone chay MOT lan tren M anh phang, roi `index_add_` gom ve `[B, d]`:

    from mv_pool import MultiViewPool, collate_studies, StudyModel

    model = StudyModel(your_backbone, d=768, n_labels=13, mode='fixed')
    imgs, seg, vf, w, y, B = batch          # tu collate_studies
    logits = model(imgs, seg, B, vf=vf, w=w)

Doi arm = doi vector `w` truyen vao. Model KHONG doi mot dong nao. Do la ly do bo nay
xuat trong so PER-IMAGE: no khop thang voi batching phang, khong phai reindex gi.
Arm ATTN: `mode='attn'`, bo `w`, truyen `vf`.

Dataset tra ve moi study mot dict; `w` lay tu `arm_weights.npz` theo `row` cua anh:

    class StudyDS(torch.utils.data.Dataset):
        def __init__(self, man, sti, w_arm, Y, split):
            self.g = {k: v for k, v in man.groupby('study_row')}
            self.rows = sti.index[(sti.split == split) & sti.keep].to_numpy()
            self.w, self.Y = w_arm, Y
        def __getitem__(self, i):
            s = self.rows[i]; m = self.g[s]
            return dict(images=torch.stack([load(d) for d in m.dicom_id]),
                        is_frontal=torch.as_tensor((m.view_group == 'frontal').values),
                        w=torch.as_tensor(self.w[m.row.values]),
                        y=torch.as_tensor(self.Y[s]))

    dl = DataLoader(ds, batch_size=64, collate_fn=collate_studies, shuffle=True)

Neu backbone DONG BANG: thay `images` bang `m.row.values` va dung
`FrozenFeatureBackbone(F)` -- luc do ca ablation chay tren CPU cung duoc, moi arm vai phut.

### Gan o dau trong kien truc

| muc | duong di | ghi chu |
|---|---|---|
| **embedding** (bo nay gia dinh) | backbone -> f -> pool -> head | head nhin vector da pool. La cau hoi CHUA ai do. |
| **xac suat** | backbone -> head -> p moi anh -> pool p | ap DUNG `w` do len `p` thay vi `f`. Khong doi model. Da do roi, ket qua null (xem cuoi). |
| **token** | ghep patch token cua moi view thanh mot chuoi -> attention | bieu dien manh nhat, dat nhat. |

Mot tinh chat tien: cung bo `w` nay dung duoc o CA muc embedding lan muc xac suat, vi
ca hai deu la to hop loi voi trong so tong bang 1. Muon do muc xac suat truoc cho re
thi khong can sua model gi ca.

## Nhung cho de sai

1. **Chuan hoa feature o muc TUNG ANH, dung o muc study.** Voi trong so tong bang 1,
   `pool(standardize(f)) == standardize(pool(f))` chinh xac, nen moi arm nam chung mot
   thang va khong arm nao thiet vi scale. Chuan hoa sau khi pool thi moi arm phai fit
   `mean/std` rieng, va ban khong con phan biet duoc "arm tot hon" voi "arm hop LR hon".
   `mean/std` lay tu anh cua study TRAIN.

2. **`FRONTAL` va study chi co lateral.** Xem `frontal_is_fallback` trong
   `study_index.parquet`. Mac dinh: cac study do dung lateral mean de giu N khong doi
   giua cac arm. Muon theo quy uoc nb04b (loai han) thi loc cot do o CA 5 arm, dung chi
   loc o FRONTAL — khong thi FRONTAL duoc cham tren tap study khac.

3. **Chia split theo `subject_id`, dung theo study.** Cot `split` co san da lam dung
   dieu do; neu ban chia lai thi phai giu nguyen rang buoc, khong thi anh cung mot
   benh nhan roi vao ca train lan test.

4. **Dung so sanh arm bang mot seed.** Chenh lech giua cac cach pooling o bo du lieu nay
   rat nho. Chay >= 3 seed va so Delta voi do lech chuan giua cac seed truoc da; Delta
   nho hon nhieu train thi khong phai phat hien.

5. **`batch_size` gio dem STUDY, khong dem anh.** Backbone thuc su nhan `M ~ 1.64 x B`
   anh (toi da 8x). Neu truoc do ban batch theo anh thi phai ha `B` xuong ~60%, khong
   thi OOM ngay batch dau. Do dai batch cung dao dong theo study -- binh thuong.

6. **Dung de anh cua cung mot study roi vao hai batch.** `collate_studies` lo viec do
   neu Dataset tra theo study; neu ban tu viet sampler thi phai giu rang buoc nay.

7. **BatchNorm trong backbone.** Batch bay gio la ANH, nen study nhieu anh se "nang"
   hon trong thong ke BN. Khong sai, nhung la mot confound nho giua cac arm neu backbone
   train cung. LayerNorm/GroupNorm khong dinh van de nay.

8. **TTA: khong lat anh.** Lat ngang doi ben tim/phoi -- voi CXR do la thay doi benh ly,
   khong phai augmentation. Chi dung bien thien thu nhan thuc te (xoay nho, crop/scale).

## Doi chieu

Do o MUC XAC SUAT (model dong bang, chi doi cach tron dau ra), pooling hoc duoc
tung cho `+0.00047 macro AUPRC, CI95 [-0.00096, +0.00199]` — chua loai 0. Tuc la
o truc nay du dia rat mong. Hay coi do la ky vong co so.
"""
with open(os.path.join(EXP_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(README)

print('files trong', EXP_DIR, ':')
tot = 0
for root, _, fs in os.walk(EXP_DIR):
    for f in sorted(fs):
        p = os.path.join(root, f); sz = os.path.getsize(p); tot += sz
        print(f'  {os.path.relpath(p, EXP_DIR):28s} {sz/1e6:9.2f} MB')
print(f'  {"TONG":28s} {tot/1e6:9.2f} MB')

## C6-bis · `mv_pool.py` — module drop-in gửi kèm

Phần khó khi gắn multiview vào một kiến trúc **không** phải công thức pooling — nó là
**batching**. Dataloader thường trả một *ảnh* một sample; multiview cần một *study* một
sample, với số ảnh thay đổi từng study.

Module này chọn cách **phẳng + segment id** thay vì đệm `[B, K, 3, H, W]`:

```
batch = 64 study  ->  M ~ 105 anh (trung binh 1.64 anh/study)
      dem toi K=8 :  512 anh  ->  lang phi ~79% compute cua backbone
      phang       :  105 anh  ->  khong lang phi gi
```

Backbone chạy **một lần** trên `M` ảnh phẳng, rồi `index_add_` gom về `[B, d]`. Đổi arm
chỉ là đổi vector `w` truyền vào — không đụng gì tới model. Đó là lý do bundle xuất trọng
số *per-image*: nó khớp thẳng với batching phẳng, không phải reindex gì cả.

In [ ]:
MV_POOL = r'''"""
mv_pool.py -- gop feature nhieu anh cua mot study thanh mot vector study-level.
Di kem bo input ablation multi-view (MIMIC-CXR cohort_v2). Chi can torch.

Dau vao PHANG (khong dem):
    f   [M, d]  feature cua M anh trong batch (M = TONG so anh, khong phai so study)
    seg [M]     seg[i] = chi so study (0..B-1) ma anh i thuoc ve
    vf  [M]     1.0 neu frontal, 0.0 neu lateral
    w   [M]     chi che do 'fixed': trong so per-image lay tu arm_weights.npz
"""
import torch
import torch.nn as nn


def segment_softmax(s, seg, B):
    """softmax cua s trong tung study. On dinh so: tru max theo tung study truoc."""
    m = torch.full((B,), float('-inf'), device=s.device, dtype=s.dtype)
    m = m.scatter_reduce(0, seg, s, reduce='amax', include_self=True)
    e = torch.exp(s - m[seg])
    den = torch.zeros(B, device=s.device, dtype=s.dtype).index_add_(0, seg, e)
    return e / den[seg].clamp(min=1e-12)


class MultiViewPool(nn.Module):
    """mode='fixed': trong so cho san, KHONG co tham so.
       mode='attn' : gated attention (Ilse et al. 2018), hoc duoc.

    Score cua 'attn' doc them co frontal/lateral -> bieu dien duoc CA 5 arm co dinh
    (chi can hoc mot hang so theo view). Giu diem nay, khong thi 'attn' la mot ho khac
    han va so sanh voi cac arm co dinh mat y nghia.

    `self.w` init bang 0 -> luc khoi dong attention deu tuyet doi, tuc dung bang arm
    MEAN. Moi sai lech khoi trung binh la thu no phai tu hoc. Giu diem nay de doc duoc
    'ATTN - MEAN' nhu phan attention thuc hoc, tach khoi phan chi do them tham so.
    """

    def __init__(self, d, mode='fixed', hidden=128):
        super().__init__()
        assert mode in ('fixed', 'attn'), mode
        self.mode, self.d = mode, d
        if mode == 'attn':
            self.V = nn.Linear(d + 2, hidden)
            self.U = nn.Linear(d + 2, hidden)
            self.w = nn.Linear(hidden, 1)
            nn.init.zeros_(self.w.weight)
            nn.init.zeros_(self.w.bias)

    def forward(self, f, seg, B, vf=None, w=None, return_attn=False):
        if self.mode == 'fixed':
            assert w is not None, "mode='fixed' can trong so w tu arm_weights.npz"
            a = w
        else:
            assert vf is not None, "mode='attn' can co view vf"
            z = torch.cat([f, vf.unsqueeze(-1), 1.0 - vf.unsqueeze(-1)], dim=-1)
            s = self.w(torch.tanh(self.V(z)) * torch.sigmoid(self.U(z))).squeeze(-1)
            a = segment_softmax(s, seg, B)
        out = torch.zeros(B, self.d, device=f.device, dtype=f.dtype)
        out.index_add_(0, seg, f * a.unsqueeze(-1))
        return (out, a) if return_attn else out


def collate_studies(batch):
    """batch: list cac dict(images=[K_i,3,H,W], is_frontal=[K_i], w=[K_i], y=[C]).
    Tra ve (imgs [M,3,H,W], seg [M], vf [M], w [M], y [B,C], B)."""
    ks   = torch.tensor([len(b['images']) for b in batch])
    imgs = torch.cat([b['images'] for b in batch], 0)
    seg  = torch.repeat_interleave(torch.arange(len(batch)), ks)
    vf   = torch.cat([b['is_frontal'] for b in batch]).float()
    w    = torch.cat([b['w'] for b in batch]).float()
    y    = torch.stack([b['y'] for b in batch])
    return imgs, seg, vf, w, y, len(batch)


class StudyModel(nn.Module):
    """backbone(per-image) -> MultiViewPool -> head(study-level).

    Doi arm = doi `w` truyen vao (hoac mode='attn'). Model khong doi mot dong nao.
    """

    def __init__(self, backbone, d, n_labels, mode='fixed', hidden_head=256, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.pool = MultiViewPool(d, mode)
        self.head = nn.Sequential(
            nn.Linear(d, hidden_head), nn.LayerNorm(hidden_head), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_head, n_labels))

    def forward(self, imgs, seg, B, vf=None, w=None):
        f = self.backbone(imgs)              # MOT lan forward cho toan bo M anh
        z = self.pool(f, seg, B, vf=vf, w=w)
        return self.head(z)


class FrozenFeatureBackbone(nn.Module):
    """Neu feature da tinh san (dong bang backbone): tra cuu thay vi forward.

    `rows` la chi so hang trong image_manifest.parquet cua tung anh trong batch.
    Ca ablation khi do chay tren CPU cung duoc, moi arm vai phut.
    """

    def __init__(self, table):
        super().__init__()
        self.register_buffer('table', torch.as_tensor(table), persistent=False)

    def forward(self, rows):
        return self.table[rows]
'''


with open(os.path.join(EXP_DIR, 'mv_pool.py'), 'w', encoding='utf-8') as f:
    f.write(MV_POOL)

# ---------------- COÅNG: module chay dung truoc khi gui di ----------------
try:
    import torch
    _ns = {}
    exec(compile(MV_POOL, 'mv_pool.py', 'exec'), _ns)
    _MVP, _seg_sm = _ns['MultiViewPool'], _ns['segment_softmax']

    _B, _d = 48, 8
    _ks  = np.array([1, 2, 3, 1] * (_B // 4)); assert len(_ks) == _B
    _seg = torch.repeat_interleave(torch.arange(_B), torch.as_tensor(_ks))
    _M   = int(_ks.sum())
    _f   = torch.randn(_M, _d)
    _vf  = (torch.rand(_M) > 0.5).float()

    with torch.no_grad():
        # (1) fixed voi w = 1/K phai bang trung binh trong tung study
        _w   = torch.as_tensor(np.repeat(1.0 / _ks, _ks), dtype=torch.float32)
        _got = _MVP(_d, 'fixed')(_f, _seg, _B, w=_w)
        _exp = torch.stack([_f[_seg == i].mean(0) for i in range(_B)])
        _e1  = float((_got - _exp).abs().max())

        # (2) attention luc init phai bang dung ket qua tren
        _got2 = _MVP(_d, 'attn')(_f, _seg, _B, vf=_vf)
        _e2   = float((_got2 - _exp).abs().max())

        # (3) segment_softmax phai tong bang 1 tren tung study
        _a  = _seg_sm(torch.randn(_M), _seg, _B)
        _e3 = float((torch.zeros(_B).index_add_(0, _seg, _a) - 1).abs().max())

    print(f'COÅNG · mv_pool fixed(1/K) == mean : {_e1:.2e}  {"OK" if _e1 <= 1e-6 else "FAIL"}')
    print(f'COÅNG · mv_pool attn @init == mean : {_e2:.2e}  {"OK" if _e2 <= 1e-6 else "FAIL"}')
    print(f'COÅNG · segment_softmax tong = 1   : {_e3:.2e}  {"OK" if _e3 <= 1e-6 else "FAIL"}')
    assert max(_e1, _e2, _e3) <= 1e-6, 'mv_pool.py SAI -> dung gui di'
except ImportError:
    print('! khong co torch trong runtime nay -> mv_pool.py da ghi nhung CHUA duoc kiem tra')

## C7 · Cổng cuối — nạp lại từ đĩa và dựng lại một arm

Đọc bundle **như bên nhận sẽ đọc** (chỉ từ file, không đụng biến trong bộ nhớ), rồi dùng
đúng đoạn code trong `README` để dựng lại một arm từ feature giả. Cổng này bắt được lỗi
serialise, lệch trật tự hàng, và `README` viết sai — ba thứ mà mọi cổng phía trên đều
không nhìn thấy.

In [ ]:
man2 = pd.read_parquet(os.path.join(EXP_DIR, 'image_manifest.parquet'))
sti2 = pd.read_parquet(os.path.join(EXP_DIR, 'study_index.parquet'))
w2   = np.load(os.path.join(EXP_DIR, 'arm_weights.npz'))
r2   = np.load(os.path.join(EXP_DIR, 'ragged.npz'))
l2   = np.load(os.path.join(EXP_DIR, 'labels.npz'), allow_pickle=True)

assert len(man2) == len(IMG) and len(sti2) == N
assert (man2.dicom_id.to_numpy() == IMG.dicom_id.to_numpy()).all()
assert (sti2.study_id.to_numpy() == SID).all()
assert l2['Y'].shape == (N, C) and list(l2['labels']) == LABELS

rng   = np.random.default_rng(0)
Ffake = rng.standard_normal((len(man2), 16)).astype('float32')
offs2 = w2['offsets']

for arm in W:
    x_disk = np.add.reduceat(Ffake * w2[f'w_{arm}'][:, None], offs2[:-1], axis=0)
    x_mem  = np.add.reduceat(Ffake * W[arm][:, None],        OFFS[:-1],  axis=0)
    e = float(np.abs(x_disk - x_mem).max())
    assert e == 0.0, f'{arm}: bundle tren dia khac trong bo nho ({e:.2e})'

# ragged: pool deu bang mask phai bang dung arm MEAN
Eg   = Ffake[r2['pad_idx']]
m    = r2['pad_mask'][:, :, None]
x_rg = (Eg * m).sum(1) / m.sum(1)
x_mean = np.add.reduceat(Ffake * W['MEAN'][:, None], OFFS[:-1], axis=0) if 'MEAN' in W \
         else np.add.reduceat(Ffake * (1.0 / n_img[srow]).astype('float32')[:, None], OFFS[:-1], axis=0)
e_rg = float(np.abs(x_rg - x_mean).max())
print(f'COÅNG · round-trip 5 arm tu dia: OK')
print(f'COÅNG · ragged pool deu == arm MEAN: max_abs_diff {e_rg:.2e}  '
      f'{"OK" if e_rg <= 1e-5 else "FAIL"}')
assert e_rg <= 1e-5, 'bo ragged khong nhat quan voi trong so -> ATTN se lech khoi cac arm kia'

if MAKE_ZIP:
    zp = shutil.make_archive(EXP_DIR, 'zip', EXP_DIR)
    print(f'\nGUI FILE NAY: {zp}  ({os.path.getsize(zp)/1e6:.1f} MB)')
print('\nBundle san sang. Bao ben nhan doc README.md truoc, nhat la muc "Nhung cho de sai".')